In [ ]:
!pip install -q torch torchaudio transformers datasets librosa jiwer accelerate
!pip install -q torchcodec


In [ ]:
import torch
import torchaudio
import librosa
import numpy as np
from datasets import load_dataset
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Using device: cpu


In [ ]:
asr_pipeline = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-small",
    device=0 if DEVICE == "cuda" else -1
)


Device set to use cpu


In [ ]:
intent_dataset = load_dataset("snips_built_in_intents")

print(intent_dataset)


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 328
    })
})


In [ ]:
intent_classifier = pipeline(
    task="text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if DEVICE == "cuda" else -1
)


Device set to use cpu


In [ ]:
sentiment_analyzer = pipeline(
    task="sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    device=0 if DEVICE == "cuda" else -1
)


Device set to use cpu


In [ ]:
SENTIMENT_LABEL_MAP = {
    "LABEL_0": "Negative",
    "LABEL_1": "Neutral",
    "LABEL_2": "Positive"
}

INTENT_LABEL_MAP = {
    "NEGATIVE": "Not clearly positive (placeholder intent)",
    "POSITIVE": "Positive / affirmative (placeholder intent)"
}


In [ ]:
def analyze_voice(audio):
    # Speech to text
    transcript = asr_pipeline(audio)["text"]

    # Intent detection (placeholder)
    intent_raw = intent_classifier(transcript)[0]
    intent_readable = INTENT_LABEL_MAP.get(
        intent_raw["label"],
        intent_raw["label"]
    )

    # Sentiment analysis
    sentiment_raw = sentiment_analyzer(transcript)[0]
    sentiment_readable = SENTIMENT_LABEL_MAP.get(
        sentiment_raw["label"],
        sentiment_raw["label"]
    )

    return {
        "transcript": transcript,
        "intent": intent_readable,
        "intent_confidence": round(intent_raw["score"], 3),
        "sentiment": sentiment_readable,
        "sentiment_confidence": round(sentiment_raw["score"], 3)
    }


In [ ]:
sample_dataset = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy",
    "clean",
    split="validation"
)

audio_sample = sample_dataset[0]["audio"]

transcription = asr_pipeline(audio_sample)["text"]
print("Transcription:")
print(transcription)


Transcription:
 Mr. Quilter is the Apostle of the Middle Classes, and we are glad to welcome his Gospel.


In [ ]:
result = analyze_voice(audio_sample)

print("Final Output:")
for k, v in result.items():
    print(f"{k}: {v}")


Final Output:
transcript:  Mr. Quilter is the Apostle of the Middle Classes, and we are glad to welcome his Gospel.
intent: Positive / affirmative (placeholder intent)
intent_confidence: 1.0
sentiment: Positive
sentiment_confidence: 0.962


In [ ]:
import json

with open("voice_analysis_output.json", "w") as f:
    json.dump(result, f, indent=4)

print("Output saved to voice_analysis_output.json")

Output saved to voice_analysis_output.json


In [ ]:
print("Speech-Driven Intent and Sentiment Analysis Pipeline Completed Successfully.")


Speech-Driven Intent and Sentiment Analysis Pipeline Completed Successfully.


In [ ]:
result = analyze_voice("Recording.wav")
print("Sentiment:", result["sentiment"])
print("Final Output:")
for k, v in result.items():
    print(f"{k}: {v}")

Sentiment: Neutral
Final Output:
transcript:  Hello, my name is Mohammad Saifali and I am talking from the XYZ company. So, do you have any time to talk with me?
intent: Not clearly positive (placeholder intent)
intent_confidence: 0.968
sentiment: Neutral
sentiment_confidence: 0.926


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
import os

SAVE_DIR = "/content/drive/MyDrive/voice_ai_models"
os.makedirs(SAVE_DIR, exist_ok=True)


In [ ]:
# ===============================
# Save ASR (Whisper) model
# ===============================
asr_pipeline.model.save_pretrained(f"{SAVE_DIR}/asr_model")
asr_pipeline.tokenizer.save_pretrained(f"{SAVE_DIR}/asr_model")

# ===============================
# Save Intent model
# ===============================
intent_classifier.model.save_pretrained(f"{SAVE_DIR}/intent_model")
intent_classifier.tokenizer.save_pretrained(f"{SAVE_DIR}/intent_model")

# ===============================
# Save Sentiment model
# ===============================
sentiment_analyzer.model.save_pretrained(f"{SAVE_DIR}/sentiment_model")
sentiment_analyzer.tokenizer.save_pretrained(f"{SAVE_DIR}/sentiment_model")

print("✅ All models successfully saved to Google Drive")


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


✅ All models successfully saved to Google Drive
